In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ArrowSpace — Best Config Evaluation                            ║
# ║  normalised · eps=0.35 · k=30 · topk=15                        ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── CELL 1 · Imports & Paths ───────────────────────────────────────────────────
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from arrowspace import ArrowSpaceBuilder
import time

ROOT       = Path.cwd().parent
CORPUS_DIR = ROOT / "data" / "nomic_emb"
QUERY_DIR  = ROOT / "data" / "nomic_bench"
BENCH_FILE = ROOT / "data" / "benchmark" / "benchmark_queries_01.json"
OUT_DIR    = CORPUS_DIR / "best_run"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = CORPUS_DIR / "embeddings_nomic_full_768d.npy"
QUERY_PATH  = QUERY_DIR  / "queries_emb_nomic_768d.npy"

# ── Best config (from grid search) ────────────────────────────────────────────
CFG = dict(eps=0.35, k=30, topk=15, p=2.0, sigma=None)
ALPHA_SEARCH = 0.75
TOP_K        = 12
PROFILES     = ["q_medium", "q_sentence", "q_verbose"]

print("Paths OK:", all(p.exists() for p in [CORPUS_PATH, QUERY_PATH, BENCH_FILE]))

In [ ]:
# ── CELL 2 · Load data ─────────────────────────────────────────────────────────
corpus    = np.load(CORPUS_PATH).astype(np.float64)
queries   = np.load(QUERY_PATH).astype(np.float64)

with open(BENCH_FILE) as f:
    benchmark = json.load(f)

print(f"Corpus  : {corpus.shape}   dtype={corpus.dtype}")
print(f"Queries : {queries.shape}  dtype={queries.dtype}")
print(f"Bench   : {len(benchmark)} entries")

In [ ]:
# ── CELL 3 · Build ArrowSpace index ───────────────────────────────────────────
print(f"Building index …  config: {CFG}")
t0 = time.perf_counter()

aspace, gl = (
    ArrowSpaceBuilder()
    .with_seed(42)
    .with_dims_reduction(enabled=False, eps=None)
    .with_sampling("simple", 1.0)
).build_and_store(CFG, corpus)

build_time = time.perf_counter() - t0
print(f"✓ Index built in {build_time:.1f}s")

In [ ]:
# ── CELL 4 · Scoring helpers ───────────────────────────────────────────────────
def score_single(
    results: list[tuple[int, float]],
    tier1: list[int],
    tier2: list[int],
    k: int = TOP_K,
) -> dict:
    if not results or all(s == 0.0 for _, s in results):
        return dict(hit=False, rank=float("nan"), mrr=0.0, ndcg=0.0,
                    recall_tier2=0.0, top1_score=0.0, score_gap=0.0, lambda0=True)

    ranked    = [r[0] for r in results[:k]]
    tier1_set = set(tier1)
    tier2_set = set(tier2)

    rank = next((i + 1 for i, p in enumerate(ranked) if p in tier1_set), None)
    hit  = rank is not None
    mrr  = 1.0 / rank if rank else 0.0

    gains = [2 if p in tier1_set else (1 if p in tier2_set else 0) for p in ranked]
    dcg   = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sorted([2] * len(tier1) + [1] * len(tier2), reverse=True)[:k]
    idcg  = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    ndcg  = dcg / idcg if idcg > 0 else 0.0

    rec2 = len(tier2_set & set(ranked)) / len(tier2_set) if tier2_set else 0.0

    return dict(hit=hit, rank=float(rank) if rank else float("nan"),
                mrr=mrr, ndcg=ndcg, recall_tier2=rec2,
                top1_score=results[0][1],
                score_gap=(results[0][1] - results[-1][1]) if len(results) > 1 else 0.0,
                lambda0=False)

print("Scoring helpers defined ✓")

In [ ]:
# ── CELL 5 · Full evaluation on ALL benchmark queries ─────────────────────────
rows          = []
lambda0_count = 0
search_errors = 0

t0 = time.perf_counter()
for bq_idx, bq in enumerate(benchmark):
    tier1 = bq["relevant_pos"]["tier1"]
    tier2 = bq["relevant_pos"]["tier2"]

    for p_idx, profile in enumerate(PROFILES):
        q_vec = queries[bq_idx, p_idx]

        try:
            results = aspace.search(q_vec, gl, ALPHA_SEARCH)
        except Exception as e:
            results = []
            search_errors += 1

        m = score_single(results, tier1, tier2)
        if m["lambda0"]:
            lambda0_count += 1
        rows.append({"bq_idx": bq_idx, "profile": profile, **m})

eval_time = time.perf_counter() - t0
df = pd.DataFrame(rows)

print(f"Evaluated {len(benchmark)} queries × {len(PROFILES)} profiles = {len(df)} rows")
print(f"Eval time : {eval_time:.1f}s   ({eval_time/len(df)*1000:.1f} ms/query)")
print(f"λ=0 events: {lambda0_count} ({lambda0_count/len(df):.2%})")
print(f"Errors    : {search_errors}")

In [ ]:
# ── CELL 6 · Aggregate metrics ─────────────────────────────────────────────────
df_valid = df[~df["lambda0"]]

sep = "─" * 52
print(sep)
print(f"  BEST RUN  —  eps={CFG['eps']}  k={CFG['k']}  topk={CFG['topk']}")
print(f"  Embeddings : nomic_full_768d  (normalised)")
print(sep)

overall = {
    "Hit@12"       : df_valid["hit"].mean(),
    "MRR"          : df_valid["mrr"].mean(),
    "NDCG@12"      : df_valid["ndcg"].mean(),
    "Recall@tier2" : df_valid["recall_tier2"].mean(),
    "Score gap"    : df_valid["score_gap"].mean(),
    "Mean rank"    : df_valid["rank"].mean(),
}
for k, v in overall.items():
    print(f"  {k:<16}: {v:.4f}")

print(sep)
print("\n  Per-profile breakdown:")
profile_metrics = (
    df_valid.groupby("profile")[["hit", "mrr", "ndcg", "recall_tier2"]]
    .mean()
    .rename(columns={"hit": "Hit@12", "mrr": "MRR", "ndcg": "NDCG@12", "recall_tier2": "Recall@T2"})
)
print(profile_metrics.round(4).to_string())
print(sep)
print(f"\n  λ=0 queries : {lambda0_count} / {len(df)}  ({lambda0_count/len(df):.2%})")
print(f"  Build time  : {build_time:.1f}s   |   Eval time: {eval_time:.1f}s")

In [ ]:
# ── CELL 7 · Per-query detail table ───────────────────────────────────────────
# Aggregate across profiles per query (mean)
per_query = (
    df_valid.groupby("bq_idx")[["hit", "mrr", "ndcg", "recall_tier2", "top1_score"]]
    .mean()
    .reset_index()
)
per_query["lambda0_any"] = df.groupby("bq_idx")["lambda0"].any().values

# Attach query text if available in benchmark
if "query" in benchmark[0]:
    per_query["query_text"] = [benchmark[i]["query"][:80] for i in per_query["bq_idx"]]

print(f"Per-query table shape: {per_query.shape}")
per_query.sort_values("ndcg", ascending=False).head(10)

In [ ]:
# ── CELL 8 · Worst queries (failures to diagnose) ─────────────────────────────
worst = per_query.sort_values("ndcg").head(20)
print("20 lowest-NDCG queries:")
display(worst.round(4))

In [ ]:
# ── CELL 9 · Score distribution plots ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"Score distributions — eps={CFG['eps']} k={CFG['k']}  (nomic 768d normalised)", fontsize=13)

for ax, col, label in zip(
    axes,
    ["ndcg", "mrr", "recall_tier2"],
    ["NDCG@12", "MRR", "Recall@Tier2"],
):
    for profile in PROFILES:
        sub = df_valid[df_valid["profile"] == profile][col]
        ax.hist(sub, bins=25, alpha=0.55, label=profile)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel("# queries")
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

plt.tight_layout()
plt.savefig(OUT_DIR / "score_distributions.png", dpi=150)
plt.show()
print(f"Saved → {OUT_DIR / 'score_distributions.png'}")

In [ ]:
# ── CELL 10 · MRR rank distribution ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
hits = df_valid[df_valid["hit"] == True]["rank"].dropna()
rank_counts = hits.value_counts().sort_index()

ax.bar(rank_counts.index, rank_counts.values, color="steelblue", edgecolor="white")
ax.set_title(f"Hit rank distribution  (Hit@12={df_valid['hit'].mean():.3f})")
ax.set_xlabel("Rank of first relevant result")
ax.set_ylabel("# queries")
ax.set_xticks(range(1, TOP_K + 1))
plt.tight_layout()
plt.savefig(OUT_DIR / "rank_distribution.png", dpi=150)
plt.show()
print(f"Saved → {OUT_DIR / 'rank_distribution.png'}")

In [ ]:
# ── CELL 11 · Save full results ────────────────────────────────────────────────
df.to_csv(OUT_DIR / "best_run_all_rows.csv", index=False)
per_query.to_csv(OUT_DIR / "best_run_per_query.csv", index=False)

summary = {
    "config"       : CFG,
    "embedding"    : "nomic_full_768d_normalised",
    "n_queries"    : len(benchmark),
    "n_rows"       : len(df),
    "hit_rate"     : float(df_valid["hit"].mean()),
    "mrr"          : float(df_valid["mrr"].mean()),
    "ndcg"         : float(df_valid["ndcg"].mean()),
    "recall_tier2" : float(df_valid["recall_tier2"].mean()),
    "lambda0_count": lambda0_count,
    "lambda0_pct"  : lambda0_count / len(df),
    "build_time_s" : build_time,
    "eval_time_s"  : eval_time,
    "per_profile"  : profile_metrics.round(4).to_dict(),
}
with open(OUT_DIR / "best_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✓ Saved:")
for p in OUT_DIR.iterdir():
    print(f"   {p.name}")